## Current research direction — HRI adaptation

Reusable preparation: review and adapt these steps for endemic-bird HRI assessment. Any legacy forest-loss target analysis in this notebook is exploratory, not an HRI definition.

The original code and saved outputs are retained. No cells were executed during restoration. Use `docs/methodology/hri_workflow.md` for the current direction and `notebooks/hri/` for the simple lecturer-facing versions. Do not describe existing forest-loss outputs as habitat-risk classes.

# 06 — Forest evidence across the Sri Lanka grid

This step prepares **threshold-free evidence**, not finalized model labels. Six pinned Hansen v1.12 TIFFs supply canopy, loss year and land/water/no-data masks across both required tiles. See the [official data description](https://storage.googleapis.com/earthenginepartners-hansen/GFC-2024-v1.12/download.html) and `data/metadata/forest_download_manifest.json`.

The provisional 5 km grid clips to the working country boundary. Native pixel centres determine membership; pixel areas use the WGS84 ellipsoid. Boundary pixels are not fractionally clipped, so small/coastal cells need review. These are mapped areas, not unbiased forest-loss area estimates. Species traits and occurrence counts are not required for this environmental evidence.

In [1]:
from pathlib import Path
import json
import runpy
import pandas as pd
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/study.json").exists())
cfg = json.loads((ROOT / "configs/study.json").read_text())
grid_id = f"lka_laea_{int(cfg['grid_size_m'])}m"
manifest = json.loads((ROOT / "data/metadata/forest_download_manifest.json").read_text())
assert len(manifest) == 6 and all(r["provider_checksum_verified"] for r in manifest)
print("Six input files verified at acquisition; run download_forest_inputs.py to recheck provider checksums.")

Six input files verified at acquisition; run download_forest_inputs.py to recheck provider checksums.


## Build per-cell evidence

The sparse histogram preserves mapped land area by **canopy percentage × loss-year code**. This lets later work compare justified canopy thresholds without repeating raster extraction. Code 0 means no mapped loss, while codes 1–24 represent 2001–2024. No-data and water are excluded from the land histogram and retained in separate summary columns.

For a chosen canopy threshold, reference year t and next year t+1, use only canopy-eligible land with loss code 0 or loss year after t for the denominator. The numerator uses eligible land lost in t+1. Do not expose future loss codes or this full-period histogram directly as model predictors.

In [2]:
report = runpy.run_path(str(ROOT / "scripts/summarize_forest_grid.py"))["main"]()
print(json.dumps(report, indent=2))

Summarized 500/2869 cells
Summarized 1000/2869 cells
Summarized 1500/2869 cells
Summarized 2000/2869 cells
Summarized 2500/2869 cells
{
  "grid_cells": 2869,
  "histogram_rows": 1197096,
  "source_release": "Hansen GFC 2024 v1.12",
  "status_counts": {
    "evidence_available": 2867,
    "no_pixel_centres": 2
  },
  "method": "Native pixel centres inside clipped country cells; WGS84 geodesic pixel areas; no boundary pixel fractional weighting.",
  "limitations": [
    "Mapped areas, not unbiased statistical area estimates.",
    "Small coastal cells may contain no pixel centres.",
    "Observed fraction includes water and is not a final modelling eligibility rule.",
    "No baseline canopy threshold selected; no target labels generated.",
    "Loss code zero means no mapped loss through 2024, not missing data.",
    "Grid remains provisional; historical release revisions mean hindcast interpretation."
  ]
}
{
  "grid_cells": 2869,
  "histogram_rows": 1197096,
  "source_release": "Hanse

In [3]:
folder = ROOT / "data/interim/spatial"
evidence = pd.read_csv(folder / f"{grid_id}_forest_evidence.csv")
histogram = pd.read_csv(folder / f"{grid_id}_canopy_loss_histogram.csv")
assert evidence.cell_id.is_unique
assert not histogram.duplicated(["cell_id", "canopy_percent", "loss_year_code"]).any()
assert histogram.canopy_percent.between(0,100).all()
assert histogram.loss_year_code.between(0,24).all()
assert histogram.mapped_land_area_m2.gt(0).all()
print(evidence.status.value_counts().to_string())
print(evidence.describe().to_string())
print("Cells containing mapped land loss:", histogram.loc[histogram.loss_year_code.gt(0), "cell_id"].nunique())
print("Cells with source no-data:", int(evidence.no_data_area_m2.gt(0).sum()))

status
evidence_available    2867
no_pixel_centres         2
       sampled_pixels  sampled_area_m2  no_data_area_m2  mapped_land_area_m2  persistent_water_area_m2  observed_area_fraction_of_sampled_footprint  mean_canopy_percent_on_mapped_land
count     2869.000000     2.869000e+03           2869.0         2.869000e+03              2.869000e+03                                       2867.0                         2867.000000
mean     30197.302196     2.302857e+07              0.0         2.271394e+07              3.146316e+05                                          1.0                           45.781930
std       7633.299819     5.823830e+06              0.0         5.925623e+06              1.154536e+06                                          0.0                           27.591868
min          0.000000     0.000000e+00              0.0         0.000000e+00              0.000000e+00                                          1.0                            0.000000
25%      32669.0000

## What remains before training

1. Justify baseline canopy eligibility and compare sensitivity to alternative thresholds.
2. Define minimum valid coverage, minimum eligible forest and treatment of small coastal cells; confirm spatial scale.
3. Generate cell-year labels with explicit undefined cases and inspect event prevalence by year and region.
4. Build dated predictors using information through year t only. Replace weather-source elevation if local terrain is needed.
5. Join on unique cell/year keys; audit leakage, duplicates and missingness; define spatial/temporal held-out groups before model fitting.

This evidence does not establish bird decline, habitat suitability, causation or a validated species-weighted conservation score.